## import

In [16]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('../data/checking-logs.sqlite')
schema = pd.io.sql.read_sql("PRAGMA table_info(test);", conn)
schema

,cid,name,type,notnull,dflt_value,pk
0,0,uid,TEXT,0,None,0
1,1,labname,TEXT,0,None,0
2,2,first_commit_ts,TIMESTAMP,0,None,0
3,3,first_view_ts,TIMESTAMP,0,None,0


## first ten

In [17]:
first_10 = pd.io.sql.read_sql("SELECT * FROM test LIMIT 10;", conn)
first_10

,uid,labname,first_commit_ts,first_view_ts
0,user_17,project1,2020-04-18 07:56:45.408648,2020-04-18 10:56:55.833899
1,user_30,laba04,2020-04-18 13:36:53.971502,2020-04-17 22:46:26.785035
2,user_30,laba04s,2020-04-18 14:51:37.498399,2020-04-17 22:46:26.785035
3,user_14,laba04,2020-04-18 15:14:00.312338,2020-04-18 10:53:52.623447
4,user_14,laba04s,2020-04-18 22:30:30.247628,2020-04-18 10:53:52.623447
5,user_19,laba04,2020-04-20 19:05:01.297780,2020-04-21 20:30:38.034966
6,user_25,laba04,2020-04-20 19:16:50.673054,2020-05-09 23:54:54.260791
7,user_21,laba04,2020-04-21 17:48:00.487806,2020-04-22 22:40:36.824081
8,user_30,project1,2020-04-22 12:36:24.053518,2020-04-17 22:46:26.785035
9,user_21,laba04s,2020-04-22 20:09:21.857747,2020-04-22 22:40:36.824081


## join and min dif finding

In [18]:
query_min = """
    SELECT
        uid,
        MIN((deadlines - strftime('%s', first_commit_ts))/3600) as min_diff
    FROM 
        test AS t
    LEFT JOIN
        deadlines as d
    ON t.labname = d.labs
    WHERE 
        t.labname != 'project1'
    GROUP BY uid
"""
df_min = pd.io.sql.read_sql(query_min, conn)
df_min

,uid,min_diff
0,user_1,6
1,user_10,39
2,user_14,84
3,user_17,34
4,user_18,3
5,user_19,32
6,user_21,33
7,user_25,2
8,user_28,8
9,user_3,60


## same for max dif

In [19]:
query_max = """
    SELECT 
        uid,
        MAX((d.deadlines - strftime('%s', t.first_commit_ts))/3600.0) as max_diff
    FROM 
        test AS t
    LEFT JOIN
        deadlines AS d
    ON t.labname = d.labs
    WHERE
        t.labname != 'project1'
    GROUP BY uid
"""
df_max = pd.io.sql.read_sql(query_max, conn)
df_max

,uid,max_diff
0,user_1,175.556667
1,user_10,132.341944
2,user_14,200.766389
3,user_17,81.591667
4,user_18,10.973611
5,user_19,148.916111
6,user_21,126.199722
7,user_25,150.870000
8,user_28,174.853056
9,user_3,182.055278


## df for avg

In [20]:
query_avg = """
    SELECT
        AVG((d.deadlines - strftime('%s', t.first_commit_ts))/3600) as avg_diff
    FROM test AS t
    LEFT JOIN
        deadlines AS d
    ON t.labname = d.labs
    WHERE
        t.labname != 'project1'
"""
df_avg = pd.io.sql.read_sql(query_avg, conn)
df_avg

,avg_diff
0,89.125


## correlation

In [21]:
query_corr = """
    SELECT
        t.uid,
        ((strftime('%s', first_commit_ts) - d.deadlines)/3600) as avg_diff,
        (
            SELECT COUNT(*)
            FROM pageviews
            WHERE uid = t.uid
        ) as pageviews
    FROM 
        test as t
    JOIN
        deadlines as d 
        ON t.labname = d.labs
    WHERE
        labname != 'project1'
    GROUP BY
        t.uid
"""
views_diff = pd.io.sql.read_sql(query_corr, conn)
views_diff[['avg_diff', 'pageviews']].corr()


,avg_diff,pageviews
avg_diff,1.000000,-0.062967
pageviews,-0.062967,1.000000


## hypothesis confirmed

In [22]:
conn.close()